# Capstone Project — Real-Time Customer Support Intelligence Platform

This notebook builds an end-to-end system that takes in customer support tickets, checks their quality, stores them in a three-layer data lake, and uses a retrieval system to suggest ready-made answers to support agents, complete with the source article they came from.

**Data:** Everything here is synthetic — support tickets, chat data, and knowledge-base articles are all generated by code in this notebook. No external account or download is required. About 12% of the generated tickets have quality problems built in on purpose (empty messages, one-word messages, wrong categories, duplicates), so the quality-checking step actually has something real to catch.

**What this notebook delivers:**

| # | Component | Points |
|---|---|---|
| 1 | Ingestion layer — Kafka producer + consumer with schema validation | 20 |
| 2 | Delta Lakehouse — bronze/silver/gold layers with MERGE and schema enforcement | 25 |
| 3 | RAG pipeline — chunking, embeddings, vector index, hybrid search, reranking | 25 |
| 4 | Orchestration — an Airflow DAG that runs every module in order | 15 |
| 5 | Quality gate — Great Expectations checks + OpenLineage events | 15 |

Each stage is written as its own Python file inside a `pipeline/` folder. The Airflow DAG at the end imports those same files and runs them in order, so there is no duplicated code between the manual run and the orchestrated run.

**How to run this:** Use "Run All" from the top. The cells depend on each other in order (later cells use variables created by earlier ones), so running them out of order will cause errors that have nothing to do with the code itself.


## Step 0 — Install everything
Run this first. It takes a few minutes because it installs Spark, Kafka client libraries, and the embedding models.

In [ ]:

# Install every library the notebook needs, from start to finish
!pip install -q pandas loguru "pydantic>=2.0" pyarrow \
    great_expectations openlineage-python \
    kafka-python-ng \
    pyspark==3.5.1 delta-spark==3.2.0 \
    sentence-transformers qdrant-client rank_bm25 \
    faker

print("All libraries installed.")


In [ ]:

# Set up the project folders
import os
os.makedirs("pipeline", exist_ok=True)
os.makedirs("data", exist_ok=True)
os.makedirs("lakehouse/bronze", exist_ok=True)
os.makedirs("lakehouse/silver", exist_ok=True)
os.makedirs("lakehouse/gold", exist_ok=True)
os.makedirs("lineage_events", exist_ok=True)
print("Folders ready:", os.listdir("."))


## Step 1 — Generate synthetic data

Two things get generated here:
- **Knowledge base articles:** about 14 short articles covering 5 common support categories (shipping, returns, account, billing, technical issues).
- **Support tickets:** about 600 tickets, each with a customer message, an agent reply, a category, a channel, and a timestamp. About 12% of them have a quality problem injected on purpose.


In [ ]:
%%writefile pipeline/data_gen.py
"""Generates synthetic knowledge-base articles and support tickets."""
import json, random, uuid
from datetime import datetime, timedelta, UTC
from faker import Faker

fake = Faker()
random.seed(42)
Faker.seed(42)

CATEGORIES = ["Shipping", "Returns & Refunds", "Account & Login", "Payment & Billing", "Technical Issue"]

KB_ARTICLES = [
    {"id": "kb-001", "category": "Shipping", "title": "Tracking a delayed shipment",
     "content": "If your order tracking has not updated in more than 5 business days, the shipment "
                 "may be stuck at a customs or carrier hub. Ask the customer for their order number "
                 "and check the carrier portal directly. Delays over 10 business days qualify for a "
                 "reshipment or refund per policy."},
    {"id": "kb-002", "category": "Shipping", "title": "Wrong address entered at checkout",
     "content": "Once an order has shipped, the address cannot be changed. Advise the customer to "
                 "contact the carrier directly to redirect the package if it has not yet been delivered. "
                 "If the carrier cannot redirect it, treat it as a lost package."},
    {"id": "kb-003", "category": "Shipping", "title": "International customs fees",
     "content": "Customers outside the seller's home country may be charged customs duties on delivery. "
                 "These fees are set by the destination country and are not collected by us, so we cannot "
                 "refund them."},
    {"id": "kb-004", "category": "Returns & Refunds", "title": "Standard return window",
     "content": "Customers can return unused items within 30 days of delivery for a full refund to the "
                 "original payment method. Items must be in original packaging with tags attached."},
    {"id": "kb-005", "category": "Returns & Refunds", "title": "Refund processing time",
     "content": "Once a returned item is received at our warehouse, refunds are issued within 3-5 business "
                 "days. It can take an additional 2-7 business days for the bank to post the credit."},
    {"id": "kb-006", "category": "Returns & Refunds", "title": "Damaged item on arrival",
     "content": "If an item arrives damaged, ask for photos of the damage and the shipping box. Damaged "
                 "items are eligible for an immediate replacement or full refund without needing to return "
                 "the item first."},
    {"id": "kb-007", "category": "Account & Login", "title": "Password reset not arriving",
     "content": "Password reset emails can take up to 10 minutes and are often filtered to spam. If it "
                 "still has not arrived, verify the email on file matches exactly, including typos or old "
                 "email addresses."},
    {"id": "kb-008", "category": "Account & Login", "title": "Account locked after failed logins",
     "content": "Accounts lock automatically after 5 failed login attempts within 15 minutes, as a security "
                 "measure. The lock clears automatically after 30 minutes, or an agent can clear it manually "
                 "after verifying identity."},
    {"id": "kb-009", "category": "Account & Login", "title": "Merging two accounts",
     "content": "We cannot automatically merge two customer accounts. Order history from the secondary "
                 "account can be manually copied over by an agent upon request, but loyalty points cannot "
                 "be transferred."},
    {"id": "kb-010", "category": "Payment & Billing", "title": "Duplicate charge on card",
     "content": "Duplicate charges are usually a temporary authorization hold, not an actual second charge, "
                 "and they disappear within 5-7 business days. If both charges post permanently, escalate "
                 "to billing for a manual refund."},
    {"id": "kb-011", "category": "Payment & Billing", "title": "Coupon code not applying",
     "content": "Coupon codes are case-sensitive and may have a minimum order value or be limited to "
                 "specific product categories. Check the coupon's terms before assuming it is a system bug."},
    {"id": "kb-012", "category": "Payment & Billing", "title": "Invoice needed for business purchase",
     "content": "A formatted invoice with tax details can be generated from the order page under "
                 "'Download Invoice'. If the business needs a different billing name, an agent can reissue "
                 "it manually."},
    {"id": "kb-013", "category": "Technical Issue", "title": "App crashing on checkout",
     "content": "Checkout crashes are most often caused by an outdated app version or a corrupted cart "
                 "cache. Ask the customer to update the app and clear the cart before retrying."},
    {"id": "kb-014", "category": "Technical Issue", "title": "Product images not loading",
     "content": "Missing product images are usually a CDN caching issue on the customer's network. "
                 "Suggest switching networks or waiting 10-15 minutes; this is not an account-specific "
                 "problem."},
]

CUSTOMER_TEMPLATES = {
    "Shipping": [
        "My order {order} hasn't moved in tracking for over a week, where is it?",
        "I put the wrong address on order {order}, can you fix it, it already shipped",
        "Why was I charged customs fees on my order from another country?",
    ],
    "Returns & Refunds": [
        "I want to return order {order}, it's been 20 days since delivery",
        "When will I get my refund for order {order}? It's been 10 days since you received it",
        "My order {order} arrived completely smashed, box was crushed",
    ],
    "Account & Login": [
        "The password reset email never arrived, I've waited 30 minutes",
        "My account got locked after I mistyped my password a few times",
        "Can you merge my two accounts, I have points on both",
    ],
    "Payment & Billing": [
        "I see two charges on my card for order {order}, was I double charged?",
        "My coupon code SAVE20 isn't applying at checkout",
        "I need a business invoice for order {order} with my company name on it",
    ],
    "Technical Issue": [
        "The app keeps crashing every time I try to check out",
        "Product images aren't loading at all on my phone",
    ],
}

AGENT_TEMPLATES = {
    "kb-001": "I checked and your package appears delayed at a carrier hub. If it doesn't update within 10 business days total, we'll issue a reshipment or full refund.",
    "kb-002": "Since the order already shipped we can't change the address on our end. Please contact the carrier directly to request redirection; if that's not possible we'll treat it as lost.",
    "kb-003": "Customs fees are set by your country's customs authority and aren't collected by us, so unfortunately we can't refund those.",
    "kb-004": "You're within our 30-day return window, so go ahead and start the return — make sure the item is unused with tags attached.",
    "kb-005": "Your return was received and refunds are processed within 3-5 business days, plus a few extra days for your bank to post it.",
    "kb-006": "I'm sorry about that! Since the item arrived damaged, I can send a replacement right away without needing you to send it back.",
    "kb-007": "Reset emails can take up to 10 minutes and sometimes land in spam. Can you confirm the exact email on your account?",
    "kb-008": "Accounts lock after 5 failed attempts as a security measure and clear automatically after 30 minutes. I can also clear it manually right now.",
    "kb-009": "We can't merge accounts automatically, but I can manually copy the order history over. Loyalty points unfortunately can't be transferred.",
    "kb-010": "That second charge is usually just a temporary hold, not an actual charge, and it should disappear in 5-7 business days.",
    "kb-011": "Coupon codes are case-sensitive and may have a minimum order value — can you double check those details for SAVE20?",
    "kb-012": "You can download a formatted invoice from your order page. If you need a different billing name I can reissue it manually.",
    "kb-013": "Checkout crashes are usually caused by an outdated app version or a corrupted cart. Please update the app and clear your cart, then try again.",
    "kb-014": "That's typically a caching issue on your network rather than your account. Try switching networks or waiting 10-15 minutes.",
}

TEMPLATE_TO_KB = {
    0: {"Shipping": "kb-001", "Returns & Refunds": "kb-004", "Payment & Billing": "kb-010"},
    1: {"Shipping": "kb-002", "Returns & Refunds": "kb-005", "Payment & Billing": "kb-011"},
    2: {"Shipping": "kb-003", "Returns & Refunds": "kb-006", "Payment & Billing": "kb-012"},
}
ACCOUNT_TEMPLATE_TO_KB = {0: "kb-007", 1: "kb-008", 2: "kb-009"}
TECH_TEMPLATE_TO_KB = {0: "kb-013", 1: "kb-014"}


def _kb_for(category, template_idx):
    if category == "Account & Login":
        return ACCOUNT_TEMPLATE_TO_KB[template_idx]
    if category == "Technical Issue":
        return TECH_TEMPLATE_TO_KB[template_idx]
    return TEMPLATE_TO_KB[template_idx][category]


def generate_kb_articles(path="data/kb_articles.json"):
    with open(path, "w") as f:
        json.dump(KB_ARTICLES, f, indent=2)
    return KB_ARTICLES


def generate_tickets(n=600, path="data/tickets.json", bad_ratio=0.12):
    tickets = []
    start = datetime(2025, 1, 1, tzinfo=UTC)
    for i in range(n):
        category = random.choice(CATEGORIES)
        templates = CUSTOMER_TEMPLATES[category]
        t_idx = random.randrange(len(templates))
        template = templates[t_idx]
        order_no = f"ORD-{random.randint(100000, 999999)}"
        message = template.format(order=order_no) if "{order}" in template else template
        kb_id = _kb_for(category, t_idx)
        agent_response = AGENT_TEMPLATES[kb_id]
        created_at = start + timedelta(minutes=random.randint(0, 60 * 24 * 200))

        ticket = {
            "ticket_id": f"T-{uuid.uuid4().hex[:10]}",
            "customer_message": message,
            "agent_response": agent_response,
            "category": category,
            "channel": random.choice(["chat", "email", "phone"]),
            "created_at": created_at.isoformat(),
            "kb_reference": kb_id,
        }
        tickets.append(ticket)

    # Inject a batch of deliberate quality problems, so the quality gate has real work to do
    n_bad = int(n * bad_ratio)
    bad_indices = random.sample(range(n), n_bad)
    for idx_pos, i in enumerate(bad_indices):
        issue = idx_pos % 4
        if issue == 0:
            tickets[i]["customer_message"] = ""                      # empty message
        elif issue == 1:
            tickets[i]["customer_message"] = "help"                  # too short to be useful
        elif issue == 2:
            tickets[i]["agent_response"] = None                      # missing response
        elif issue == 3:
            tickets[i]["category"] = "Unknown"                       # not a real category

    # Duplicate a handful of tickets exactly, so uniqueness checks have something to catch
    for _ in range(max(1, n // 100)):
        tickets.append(dict(random.choice(tickets)))

    with open(path, "w") as f:
        json.dump(tickets, f, indent=2)
    return tickets


if __name__ == "__main__":
    generate_kb_articles()
    generate_tickets()
    print("Synthetic data generated.")


In [ ]:

import sys
sys.path.insert(0, ".")
from pipeline.data_gen import generate_kb_articles, generate_tickets

kb_articles = generate_kb_articles()
tickets = generate_tickets(n=600)

print(f"Knowledge base articles: {len(kb_articles)}")
print(f"Tickets generated: {len(tickets)}")
print("\nSample ticket:")
tickets[0]


## Step 2 — Data contract with Pydantic

Before anything gets stored, every ticket has to pass a strict contract that checks it is complete and makes sense.

In [ ]:
%%writefile pipeline/contract.py
"""Pydantic data contract for support tickets — nothing gets stored unless it passes this."""
from pydantic import BaseModel, field_validator, ConfigDict
from typing import Optional

VALID_CATEGORIES = {"Shipping", "Returns & Refunds", "Account & Login", "Payment & Billing", "Technical Issue"}


class TicketContract(BaseModel):
    model_config = ConfigDict(strict=False)

    ticket_id: str
    customer_message: str
    agent_response: str
    category: str
    channel: str
    created_at: str
    kb_reference: Optional[str] = None

    @field_validator("customer_message")
    @classmethod
    def message_required(cls, v: str) -> str:
        if not v or len(v.strip()) < 5:
            raise ValueError("customer_message is empty or too short to be a real ticket")
        return v.strip()

    @field_validator("agent_response")
    @classmethod
    def response_required(cls, v):
        if not v or not str(v).strip():
            raise ValueError("agent_response is missing")
        return str(v).strip()

    @field_validator("category")
    @classmethod
    def category_must_be_known(cls, v: str) -> str:
        if v not in VALID_CATEGORIES:
            raise ValueError(f"category '{v}' is not one of the known support categories")
        return v


def validate_batch(records: list[dict]):
    """Splits a batch of raw records into (valid, rejected_with_reason)."""
    valid, rejected = [], []
    for r in records:
        try:
            obj = TicketContract(**r)
            valid.append(obj.model_dump())
        except Exception as e:
            rejected.append({**r, "_rejection_reason": str(e)})
    return valid, rejected


In [ ]:

from pipeline.contract import validate_batch

valid_tickets, rejected_tickets = validate_batch(tickets)
print(f"Valid tickets   : {len(valid_tickets)}")
print(f"Rejected tickets: {len(rejected_tickets)}")
print("\nSample rejection reasons:")
for r in rejected_tickets[:5]:
    print(" -", r["_rejection_reason"])


## Step 3 — Ingestion layer: a real Kafka broker

Apache Kafka runs locally inside this notebook using **KRaft mode**, which means no separate ZooKeeper process is needed. Once it's running:
1. A **producer** publishes every ticket as a JSON message on a topic called `support-tickets`.
2. A **consumer** reads the messages back and passes them through the same `TicketContract` from the previous step before accepting them.

> If the Kafka download fails because of a network hiccup, just run the cell again.

In [ ]:

# Install Java (Kafka needs it) and download Kafka itself
import os, subprocess, socket, time

KAFKA_VERSION = "3.7.1"
SCALA_VERSION = "2.13"
KAFKA_DIR = f"kafka_{SCALA_VERSION}-{KAFKA_VERSION}"
TGZ = f"{KAFKA_DIR}.tgz"

subprocess.run(["apt-get", "update", "-qq"], check=False)
subprocess.run(["apt-get", "install", "-y", "-qq", "openjdk-17-jre-headless"], check=False)

if not os.path.exists(TGZ):
    # Try the main Apache mirror first, then fall back to the archive
    urls = [
        f"https://downloads.apache.org/kafka/{KAFKA_VERSION}/{TGZ}",
        f"https://archive.apache.org/dist/kafka/{KAFKA_VERSION}/{TGZ}",
    ]
    ok = False
    for url in urls:
        rc = subprocess.run(["curl", "-sSL", "-o", TGZ, url]).returncode
        if rc == 0 and os.path.getsize(TGZ) > 10_000_000:
            print(f"Downloaded Kafka from {url}")
            ok = True
            break
    if not ok:
        raise RuntimeError("Could not download Kafka from either mirror - check your internet connection and retry.")

if not os.path.exists(KAFKA_DIR):
    subprocess.run(["tar", "-xzf", TGZ], check=True)

print("Kafka files ready:", KAFKA_DIR)


In [ ]:

# Format storage for KRaft mode and start the broker
import uuid as _uuid

cluster_id = str(_uuid.uuid4())
props = f"{KAFKA_DIR}/config/kraft/server.properties"

fmt = subprocess.run(
    [f"{KAFKA_DIR}/bin/kafka-storage.sh", "format", "-t", cluster_id, "-c", props],
    capture_output=True, text=True,
)
print(fmt.stdout[-500:], fmt.stderr[-500:])

kafka_log = open("kafka.log", "w")
kafka_proc = subprocess.Popen(
    [f"{KAFKA_DIR}/bin/kafka-server-start.sh", props],
    stdout=kafka_log, stderr=subprocess.STDOUT,
)

# Wait until the broker is actually accepting connections before moving on
ready = False
for _ in range(45):
    try:
        s = socket.create_connection(("localhost", 9092), timeout=1)
        s.close()
        ready = True
        break
    except OSError:
        time.sleep(2)

if not ready:
    raise RuntimeError("Kafka did not start in time - check kafka.log for details:\n" + open("kafka.log").read()[-2000:])

print("Kafka broker is up on localhost:9092 (PID", kafka_proc.pid, ")")


In [ ]:

# Create the topic we'll publish tickets to
create = subprocess.run(
    [f"{KAFKA_DIR}/bin/kafka-topics.sh", "--create", "--if-not-exists",
     "--topic", "support-tickets", "--bootstrap-server", "localhost:9092",
     "--partitions", "3", "--replication-factor", "1"],
    capture_output=True, text=True,
)
print(create.stdout, create.stderr)

list_topics = subprocess.run(
    [f"{KAFKA_DIR}/bin/kafka-topics.sh", "--list", "--bootstrap-server", "localhost:9092"],
    capture_output=True, text=True,
)
print("Topics on the broker:", list_topics.stdout)


In [ ]:
%%writefile pipeline/ingest.py
"""Kafka producer and consumer for the support-tickets ingestion layer."""
import json
from kafka import KafkaProducer, KafkaConsumer

BOOTSTRAP = "localhost:9092"
TOPIC = "support-tickets"


def produce_tickets(tickets):
    producer = KafkaProducer(
        bootstrap_servers=BOOTSTRAP,
        value_serializer=lambda v: json.dumps(v).encode("utf-8"),
        key_serializer=lambda k: k.encode("utf-8") if k else None,
    )
    for t in tickets:
        producer.send(TOPIC, key=t.get("ticket_id"), value=t)
    producer.flush()
    producer.close()
    return len(tickets)


def consume_tickets(timeout_ms=20000):
    consumer = KafkaConsumer(
        TOPIC,
        bootstrap_servers=BOOTSTRAP,
        auto_offset_reset="earliest",
        enable_auto_commit=False,
        value_deserializer=lambda v: json.loads(v.decode("utf-8")),
        consumer_timeout_ms=timeout_ms,
    )
    consumed = [msg.value for msg in consumer]
    consumer.close()
    return consumed


In [ ]:

from pipeline.ingest import produce_tickets, consume_tickets

n_sent = produce_tickets(tickets)
print(f"Sent {n_sent} messages to the 'support-tickets' topic")

consumed_tickets = consume_tickets()
print(f"Read back {len(consumed_tickets)} messages from Kafka")

# Validate everything that actually came off the wire - this is the real raw batch for Bronze
from pipeline.contract import validate_batch
valid_tickets, rejected_tickets = validate_batch(consumed_tickets)
print(f"Valid after the Kafka round-trip   : {len(valid_tickets)}")
print(f"Rejected after the Kafka round-trip: {len(rejected_tickets)}")


## Step 4 — Delta Lakehouse: Bronze, Silver, Gold with MERGE

- **Bronze**: everything that came out of Kafka, exactly as it is, no filtering.
- **Silver**: only the tickets that passed the data contract, cleaned up and structured.
- **Gold**: an analytical rollup (ticket count and average reply length per category per day), kept up to date with **MERGE** instead of a plain append.

This step also proves **schema enforcement**: trying to write a column with the wrong data type into the Silver table gets rejected automatically by Delta.

In [ ]:
%%writefile pipeline/lakehouse.py
"""Delta Lakehouse: bronze, silver, and gold layers, kept up to date with MERGE."""
from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta.tables import DeltaTable

BRONZE_PATH = "lakehouse/bronze/support_tickets"
SILVER_PATH = "lakehouse/silver/support_tickets"
GOLD_PATH = "lakehouse/gold/category_metrics"


def get_spark():
    # Spark session configured with the Delta extensions and catalog
    builder = (
        SparkSession.builder.appName("capstone-lakehouse")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        .config("spark.sql.shuffle.partitions", "4")
    )
    return configure_spark_with_delta_pip(builder).getOrCreate()


def write_bronze(spark, records):
    # Bronze is append-only and untouched - it should keep every record it has ever seen
    df = spark.createDataFrame(records)
    df.write.format("delta").mode("append").option("mergeSchema", "true").save(BRONZE_PATH)
    return df


def upsert_silver(spark, valid_records):
    # Silver holds only contract-valid records, upserted by ticket_id
    df = (
        spark.createDataFrame(valid_records)
        .withColumn("ingested_at", F.current_timestamp())
        .dropDuplicates(["ticket_id"])  # a single incoming batch must not contain the same key twice
    )

    if not DeltaTable.isDeltaTable(spark, SILVER_PATH):
        df.write.format("delta").mode("overwrite").save(SILVER_PATH)
    else:
        target = DeltaTable.forPath(spark, SILVER_PATH)
        (
            target.alias("t")
            .merge(df.alias("s"), "t.ticket_id = s.ticket_id")
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
    return df


def upsert_gold(spark):
    # Gold is an aggregated rollup per category and day, also kept up to date with MERGE
    silver = spark.read.format("delta").load(SILVER_PATH)
    agg = (
        silver.withColumn("day", F.to_date("created_at"))
        .groupBy("category", "day")
        .agg(
            F.count("*").alias("ticket_count"),
            F.avg(F.length("agent_response")).alias("avg_response_length"),
        )
    )

    if not DeltaTable.isDeltaTable(spark, GOLD_PATH):
        agg.write.format("delta").mode("overwrite").save(GOLD_PATH)
    else:
        target = DeltaTable.forPath(spark, GOLD_PATH)
        (
            target.alias("t")
            .merge(agg.alias("s"), "t.category = s.category AND t.day = s.day")
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
    return agg


def demonstrate_schema_enforcement(spark):
    # Try writing a column with the wrong type on purpose - Delta should refuse it
    from pyspark.sql.types import StructType, StructField, IntegerType

    bad_schema = StructType([StructField("ticket_id", IntegerType())])
    bad_df = spark.createDataFrame([(123,)], schema=bad_schema)
    try:
        bad_df.write.format("delta").mode("append").save(SILVER_PATH)
        return False, "No error was raised - this is unexpected."
    except Exception as e:
        return True, str(e)[:300]


In [ ]:

from pipeline.lakehouse import get_spark, write_bronze, upsert_silver, upsert_gold, demonstrate_schema_enforcement

spark = get_spark()

# Bronze: everything Kafka handed back, untouched
write_bronze(spark, consumed_tickets)
print("Bronze row count:", spark.read.format("delta").load("lakehouse/bronze/support_tickets").count())

# Silver: only the contract-valid rows (first load)
upsert_silver(spark, valid_tickets)
print("Silver row count after first load:", spark.read.format("delta").load("lakehouse/silver/support_tickets").count())


In [ ]:

# Simulate a second, incremental batch to prove MERGE actually updates instead of just appending
from pipeline.data_gen import generate_tickets
from pipeline.contract import validate_batch
from pipeline.ingest import produce_tickets, consume_tickets

incremental_tickets = generate_tickets(n=80, path="data/tickets_batch2.json")
produce_tickets(incremental_tickets)
incremental_consumed = consume_tickets()
valid_incremental, rejected_incremental = validate_batch(incremental_consumed)

write_bronze(spark, incremental_consumed)
upsert_silver(spark, valid_incremental)

silver_count_after = spark.read.format("delta").load("lakehouse/silver/support_tickets").count()
print(f"Incremental batch: {len(valid_incremental)} valid / {len(rejected_incremental)} rejected")
print("Silver row count after the MERGE:", silver_count_after)


In [ ]:

# Gold layer: the aggregated rollup, also updated with MERGE
gold_df = upsert_gold(spark)
gold_df.orderBy("category", "day").show(20, truncate=False)


In [ ]:

# Confirm Delta actually refuses a wrong-typed write into Silver
ok, message = demonstrate_schema_enforcement(spark)
print("The write was rejected as expected:" if ok else "Warning:", message)


## Step 5 — Quality gate: Great Expectations + OpenLineage

A real **Great Expectations 1.x** checkpoint runs against the Silver layer, and real **OpenLineage** events (START/COMPLETE) get emitted for each stage of the pipeline.

In [ ]:
%%writefile pipeline/quality.py
"""Quality gate: a real Great Expectations checkpoint plus real OpenLineage events."""
import os
import great_expectations as gx
import great_expectations.expectations as gxe

from openlineage.client import OpenLineageClient
from openlineage.client.transport.file import FileConfig, FileTransport
from openlineage.client.event_v2 import RunEvent, RunState, Run, Job
from openlineage.client.uuid import generate_new_uuid
from datetime import datetime, UTC

VALID_CATEGORIES = ["Shipping", "Returns & Refunds", "Account & Login", "Payment & Billing", "Technical Issue"]


def run_ge_checkpoint(pandas_df):
    """Runs a real GX 1.x fluent-API checkpoint over the silver (contract-validated) data."""
    context = gx.get_context(mode="ephemeral")
    data_source = context.data_sources.add_pandas("pandas_support_source")
    data_asset = data_source.add_dataframe_asset(name="support_tickets")
    batch_definition = data_asset.add_batch_definition_whole_dataframe("whole_df")

    suite = context.suites.add(gx.ExpectationSuite(name="support_tickets_quality_suite"))
    suite.add_expectation(gxe.ExpectColumnValuesToNotBeNull(column="ticket_id"))
    suite.add_expectation(gxe.ExpectColumnValuesToNotBeNull(column="customer_message"))
    suite.add_expectation(gxe.ExpectColumnValuesToNotBeNull(column="agent_response"))
    suite.add_expectation(gxe.ExpectColumnValuesToBeInSet(column="category", value_set=VALID_CATEGORIES))
    suite.add_expectation(gxe.ExpectColumnValueLengthsToBeBetween(column="customer_message", min_value=5))
    suite.add_expectation(gxe.ExpectColumnValuesToBeUnique(column="ticket_id"))

    batch = batch_definition.get_batch(batch_parameters={"dataframe": pandas_df})
    result = batch.validate(suite)
    return {
        "success": bool(result.success),
        "statistics": result.statistics,
    }


class LineageEmitter:
    """Thin wrapper around the real openlineage-python client, writing events to a local file."""

    def __init__(self, job_namespace="support_capstone", log_path="lineage_events/openlineage_run.log"):
        log_dir = os.path.dirname(log_path)
        if log_dir:
            os.makedirs(log_dir, exist_ok=True)
        self.log_path = log_path
        self.transport = FileTransport(FileConfig(log_file_path=log_path))
        self.client = OpenLineageClient(transport=self.transport)
        self.job_namespace = job_namespace
        self.run_ids = {}

    def start(self, job_name, inputs=None, outputs=None):
        # Record that this stage has started
        run_id = str(generate_new_uuid())
        self.run_ids[job_name] = run_id
        self.client.emit(RunEvent(
            eventType=RunState.START,
            eventTime=datetime.now(UTC).isoformat(),
            run=Run(runId=run_id),
            job=Job(namespace=self.job_namespace, name=job_name),
            producer="capstone-support-pipeline",
            inputs=inputs or [],
            outputs=outputs or [],
        ))
        return run_id

    def complete(self, job_name, inputs=None, outputs=None):
        # Record that this stage finished successfully
        run_id = self.run_ids.get(job_name, str(generate_new_uuid()))
        self.client.emit(RunEvent(
            eventType=RunState.COMPLETE,
            eventTime=datetime.now(UTC).isoformat(),
            run=Run(runId=run_id),
            job=Job(namespace=self.job_namespace, name=job_name),
            producer="capstone-support-pipeline",
            inputs=inputs or [],
            outputs=outputs or [],
        ))

    def fail(self, job_name):
        # Record that this stage failed
        run_id = self.run_ids.get(job_name, str(generate_new_uuid()))
        self.client.emit(RunEvent(
            eventType=RunState.FAIL,
            eventTime=datetime.now(UTC).isoformat(),
            run=Run(runId=run_id),
            job=Job(namespace=self.job_namespace, name=job_name),
            producer="capstone-support-pipeline",
        ))


In [ ]:

from pipeline.quality import run_ge_checkpoint, LineageEmitter

lineage = LineageEmitter()

lineage.start("kafka_ingestion", inputs=[{"namespace": "synthetic", "name": "tickets.json"}],
              outputs=[{"namespace": "kafka", "name": "support-tickets"}])
lineage.complete("kafka_ingestion")

lineage.start("bronze_load", inputs=[{"namespace": "kafka", "name": "support-tickets"}],
              outputs=[{"namespace": "delta", "name": "lakehouse/bronze/support_tickets"}])
lineage.complete("bronze_load")

silver_pdf = spark.read.format("delta").load("lakehouse/silver/support_tickets").toPandas()

lineage.start("silver_quality_gate", inputs=[{"namespace": "delta", "name": "lakehouse/bronze/support_tickets"}],
              outputs=[{"namespace": "delta", "name": "lakehouse/silver/support_tickets"}])
ge_result = run_ge_checkpoint(silver_pdf)
if ge_result["success"]:
    lineage.complete("silver_quality_gate")
else:
    lineage.fail("silver_quality_gate")

print("Great Expectations result:", ge_result)


In [ ]:

lineage.start("gold_aggregation", inputs=[{"namespace": "delta", "name": "lakehouse/silver/support_tickets"}],
              outputs=[{"namespace": "delta", "name": "lakehouse/gold/category_metrics"}])
lineage.complete("gold_aggregation")

print("OpenLineage events were written to", lineage.log_path)

# Read the log back defensively - if this cell was run without the ones above,
# say so clearly instead of failing with a raw traceback.
import os
if os.path.exists(lineage.log_path):
    with open(lineage.log_path) as f:
        lines = f.readlines()
    print(f"Total lineage events recorded: {len(lines)}")
    print("\\nLast event:\\n", lines[-1][:400])
else:
    print("No log file found yet - make sure the cells in Step 5 were run in order, starting from 'lineage = LineageEmitter()' above.")


## Step 6 — RAG: chunking, embeddings, vector index, hybrid search, reranking

1. **Chunking**: knowledge-base articles get split into small pieces.
2. **Embedding**: each piece is turned into a vector with `sentence-transformers`.
3. **Vector index**: the vectors are stored in **Qdrant**, running in-memory with no external server.
4. **Hybrid search**: dense search (Qdrant) and keyword search (BM25) are combined with Reciprocal Rank Fusion.
5. **Reranking**: a cross-encoder reorders the top candidates for better accuracy.
6. **Answer suggestion**: when a new ticket comes in, the system suggests a ready reply along with the article it came from.

In [ ]:
%%writefile pipeline/rag.py
"""RAG: chunking, embeddings, a Qdrant vector index, BM25 hybrid search, and cross-encoder reranking."""
import re
import uuid
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi

COLLECTION = "kb_chunks"
EMBED_MODEL_NAME = "all-MiniLM-L6-v2"
RERANK_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"


def chunk_articles(kb_articles, max_sentences=2):
    """Simple sentence-window chunking - enough for these short articles."""
    chunks = []
    for article in kb_articles:
        sentences = re.split(r"(?<=[.!?])\s+", article["content"].strip())
        for i in range(0, len(sentences), max_sentences):
            window = " ".join(sentences[i:i + max_sentences]).strip()
            if not window:
                continue
            chunks.append({
                "chunk_id": str(uuid.uuid4()),
                "article_id": article["id"],
                "article_title": article["title"],
                "category": article["category"],
                "text": window,
            })
    return chunks


class RagIndex:
    def __init__(self):
        self.embed_model = SentenceTransformer(EMBED_MODEL_NAME)
        self.reranker = CrossEncoder(RERANK_MODEL_NAME)
        self.client = QdrantClient(":memory:")
        self.chunks = []
        self.bm25 = None
        self._tokenized_corpus = []

    def build(self, chunks):
        self.chunks = chunks
        texts = [c["text"] for c in chunks]

        # Dense vector index (Qdrant, in-memory)
        vectors = self.embed_model.encode(texts, normalize_embeddings=True)
        self.client.create_collection(
            collection_name=COLLECTION,
            vectors_config=VectorParams(size=vectors.shape[1], distance=Distance.COSINE),
        )
        points = [
            PointStruct(id=i, vector=vectors[i].tolist(), payload=chunks[i])
            for i in range(len(chunks))
        ]
        self.client.upsert(collection_name=COLLECTION, points=points)

        # Keyword index (BM25)
        self._tokenized_corpus = [t.lower().split() for t in texts]
        self.bm25 = BM25Okapi(self._tokenized_corpus)

    def hybrid_search(self, query, top_k=5, fusion_k=60):
        """Combines dense (Qdrant) and keyword (BM25) results with Reciprocal Rank Fusion."""
        query_vec = self.embed_model.encode(query, normalize_embeddings=True).tolist()
        response = self.client.query_points(collection_name=COLLECTION, query=query_vec, limit=20)
        dense_hits = response.points
        dense_rank = {hit.id: rank for rank, hit in enumerate(dense_hits)}

        bm25_scores = self.bm25.get_scores(query.lower().split())
        bm25_rank_order = sorted(range(len(bm25_scores)), key=lambda i: bm25_scores[i], reverse=True)[:20]
        sparse_rank = {idx: rank for rank, idx in enumerate(bm25_rank_order)}

        # Combine both rankings: each chunk earns points based on its position in each list
        all_ids = set(dense_rank) | set(sparse_rank)
        fused = []
        for cid in all_ids:
            score = 0.0
            if cid in dense_rank:
                score += 1.0 / (fusion_k + dense_rank[cid])
            if cid in sparse_rank:
                score += 1.0 / (fusion_k + sparse_rank[cid])
            fused.append((cid, score))
        fused.sort(key=lambda x: x[1], reverse=True)
        top_ids = [cid for cid, _ in fused[:top_k]]
        return [self.chunks[i] for i in top_ids]

    def rerank(self, query, candidates, top_n=3):
        # Reorder the top candidates with a cross-encoder for better precision
        pairs = [[query, c["text"]] for c in candidates]
        scores = self.reranker.predict(pairs)
        ranked = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)
        return [{**c, "rerank_score": float(s)} for c, s in ranked[:top_n]]

    def suggest_answer(self, customer_message, top_k=5, top_n=1):
        # The function a support agent would actually call: hybrid search, then rerank
        candidates = self.hybrid_search(customer_message, top_k=top_k)
        best = self.rerank(customer_message, candidates, top_n=top_n)
        return best


In [ ]:

from pipeline.rag import chunk_articles, RagIndex

chunks = chunk_articles(kb_articles)
print(f"Built {len(chunks)} chunks from {len(kb_articles)} knowledge-base articles")

rag = RagIndex()
rag.build(chunks)
print("Qdrant (in-memory) and BM25 indexes are ready.")


In [ ]:

# A brand new ticket, never seen verbatim in the knowledge base
new_ticket = "Hey, I ordered something 9 days ago and the tracking page still says 'label created', is it lost?"

suggestion = rag.suggest_answer(new_ticket, top_k=5, top_n=1)[0]

print("Customer message:", new_ticket)
print("\\nSuggested source :", suggestion["article_title"], f"(category: {suggestion['category']})")
print("Suggested snippet:", suggestion["text"])
print(f"Confidence score : {suggestion['rerank_score']:.3f}")


In [ ]:

# Two more queries to sanity-check hybrid search and reranking across categories
for q in [
    "my card got charged twice for the same order",
    "the app crashes right when I hit checkout",
]:
    top = rag.suggest_answer(q, top_k=5, top_n=1)[0]
    print(f"Q: {q}\\n -> [{top['category']}] {top['article_title']}  (score={top['rerank_score']:.3f})\\n")


## Step 7 — Orchestration: an Airflow DAG that connects every module

This builds a real Apache Airflow DAG that calls the **same `pipeline/*.py` files** used above, in this order:

`generate_data -> kafka_ingest -> lakehouse_load -> quality_gate -> build_rag_index`

It runs through `airflow dags test`, which executes the whole DAG from start to finish without needing a full webserver or scheduler running - a good fit for a notebook environment.

In [ ]:

# Install Airflow with the constraints file that matches this Python version.
# --upgrade-strategy eager forces pip to also upgrade already-installed shared
# dependencies (like typing_extensions and pydantic) to the versions Airflow's
# constraints file expects, instead of leaving older versions in place.
import sys, subprocess

py_ver = f"{sys.version_info.major}.{sys.version_info.minor}"
AIRFLOW_VERSION = "2.9.3"
constraint_url = (
    f"https://raw.githubusercontent.com/apache/airflow/constraints-{AIRFLOW_VERSION}/"
    f"constraints-{py_ver}.txt"
)
print("Using constraints file:", constraint_url)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "--upgrade-strategy", "eager",
     f"apache-airflow=={AIRFLOW_VERSION}", "--constraint", constraint_url],
    check=True,
)

# Belt-and-suspenders: make sure typing_extensions is new enough for pydantic_core
# (it needs to provide `Sentinel`, which only exists in newer releases).
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "typing_extensions>=4.12.0"],
    check=True,
)

import importlib
print("typing_extensions ok:", importlib.import_module("typing_extensions").__file__)
print("Airflow installed.")


In [ ]:

# Set AIRFLOW_HOME and initialize the metadata database
import os, shutil, subprocess

AIRFLOW_HOME = os.path.abspath("airflow_home")

# Start from a clean state in case a previous attempt left a broken config or database behind
if os.path.exists(AIRFLOW_HOME):
    shutil.rmtree(AIRFLOW_HOME)
os.makedirs(os.path.join(AIRFLOW_HOME, "dags"), exist_ok=True)

os.environ["AIRFLOW_HOME"] = AIRFLOW_HOME
os.environ["AIRFLOW__CORE__LOAD_EXAMPLES"] = "False"
os.environ["AIRFLOW__DATABASE__SQL_ALCHEMY_CONN"] = f"sqlite:///{AIRFLOW_HOME}/airflow.db"

result = subprocess.run(
    ["airflow", "db", "migrate"],
    capture_output=True, text=True, env=os.environ,
)
print(result.stdout[-3000:])
print(result.stderr[-3000:])

if result.returncode != 0:
    raise RuntimeError(
        "airflow db migrate failed - see the actual error message printed above for the real cause."
    )

print("Airflow's metadata database is ready at", AIRFLOW_HOME)


In [ ]:
%%writefile airflow_home/dags/capstone_support_dag.py
"""
Airflow DAG that runs every module of this project end to end:
generate_data -> kafka_ingest -> lakehouse_load -> quality_gate -> build_rag_index
"""
import sys, os
sys.path.insert(0, os.environ.get("CAPSTONE_PROJECT_DIR", "."))

from datetime import datetime
from airflow import DAG
from airflow.operators.python import PythonOperator


def task_generate_data(**context):
    # Stage 1: generate the synthetic data
    from pipeline.data_gen import generate_kb_articles, generate_tickets
    kb = generate_kb_articles()
    tickets = generate_tickets(n=600)
    return {"kb_count": len(kb), "ticket_count": len(tickets)}


def task_kafka_ingest(**context):
    # Stage 2: produce and consume from Kafka, then validate against the contract
    import json
    from pipeline.ingest import produce_tickets, consume_tickets
    from pipeline.contract import validate_batch

    tickets = json.load(open("data/tickets.json"))
    produce_tickets(tickets)
    consumed = consume_tickets()
    valid, rejected = validate_batch(consumed)

    json.dump(valid, open("data/dag_valid.json", "w"))
    json.dump(rejected, open("data/dag_rejected.json", "w"))
    json.dump(consumed, open("data/dag_consumed.json", "w"))
    return {"consumed": len(consumed), "valid": len(valid), "rejected": len(rejected)}


def task_lakehouse_load(**context):
    # Stage 3: load the data into the Bronze/Silver/Gold layers
    import json
    from pipeline.lakehouse import get_spark, write_bronze, upsert_silver, upsert_gold

    consumed = json.load(open("data/dag_consumed.json"))
    valid = json.load(open("data/dag_valid.json"))

    spark = get_spark()
    write_bronze(spark, consumed)
    upsert_silver(spark, valid)
    upsert_gold(spark)

    silver_count = spark.read.format("delta").load("lakehouse/silver/support_tickets").count()
    return {"silver_rows": silver_count}


def task_quality_gate(**context):
    # Stage 4: run the quality checks and record a lineage event
    from pipeline.lakehouse import get_spark
    from pipeline.quality import run_ge_checkpoint, LineageEmitter

    spark = get_spark()
    silver_pdf = spark.read.format("delta").load("lakehouse/silver/support_tickets").toPandas()

    lineage = LineageEmitter(log_path="lineage_events/airflow_run.log")
    lineage.start("dag_quality_gate")
    result = run_ge_checkpoint(silver_pdf)
    if result["success"]:
        lineage.complete("dag_quality_gate")
    else:
        lineage.fail("dag_quality_gate")
    return result


def task_build_rag_index(**context):
    # Stage 5: build the RAG index and try a sample suggestion
    import json
    from pipeline.rag import chunk_articles, RagIndex

    kb_articles = json.load(open("data/kb_articles.json"))
    chunks = chunk_articles(kb_articles)
    rag = RagIndex()
    rag.build(chunks)
    demo = rag.suggest_answer("my package tracking hasn't updated in a week", top_n=1)[0]
    return {"chunks": len(chunks), "demo_source": demo["article_title"]}


with DAG(
    dag_id="capstone_support_pipeline",
    start_date=datetime(2025, 1, 1),
    schedule=None,
    catchup=False,
    tags=["capstone", "support-intelligence"],
) as dag:

    generate_data = PythonOperator(task_id="generate_data", python_callable=task_generate_data)
    kafka_ingest = PythonOperator(task_id="kafka_ingest", python_callable=task_kafka_ingest)
    lakehouse_load = PythonOperator(task_id="lakehouse_load", python_callable=task_lakehouse_load)
    quality_gate = PythonOperator(task_id="quality_gate", python_callable=task_quality_gate)
    build_rag_index = PythonOperator(task_id="build_rag_index", python_callable=task_build_rag_index)

    generate_data >> kafka_ingest >> lakehouse_load >> quality_gate >> build_rag_index


In [ ]:

import os, subprocess

# Re-declare the Airflow environment here too, in case this cell is run after a
# kernel restart without re-running the earlier setup cells - this makes it safe
# to run standalone instead of failing with a confusing "initialize the database" error.
AIRFLOW_HOME = os.path.abspath("airflow_home")
os.environ["AIRFLOW_HOME"] = AIRFLOW_HOME
os.environ["AIRFLOW__CORE__LOAD_EXAMPLES"] = "False"
os.environ["AIRFLOW__DATABASE__SQL_ALCHEMY_CONN"] = f"sqlite:///{AIRFLOW_HOME}/airflow.db"
os.environ["CAPSTONE_PROJECT_DIR"] = os.path.abspath(".")

# If the metadata database hasn't been created yet, create it now instead of failing
if not os.path.exists(os.path.join(AIRFLOW_HOME, "airflow.db")):
    print("Metadata database not found yet - running 'airflow db migrate' first.")
    migrate = subprocess.run(["airflow", "db", "migrate"], capture_output=True, text=True, env=os.environ)
    print(migrate.stdout[-2000:])
    print(migrate.stderr[-2000:])
    if migrate.returncode != 0:
        raise RuntimeError("airflow db migrate failed - see the error message printed above.")

# Confirm Airflow can see the DAG
list_dags = subprocess.run(["airflow", "dags", "list"], capture_output=True, text=True, env=os.environ)
print(list_dags.stdout[-1500:])
print(list_dags.stderr[-1500:])


In [ ]:

import os, subprocess

# Same defensive re-declaration as the cell above, so this cell also works standalone
AIRFLOW_HOME = os.path.abspath("airflow_home")
os.environ["AIRFLOW_HOME"] = AIRFLOW_HOME
os.environ["AIRFLOW__CORE__LOAD_EXAMPLES"] = "False"
os.environ["AIRFLOW__DATABASE__SQL_ALCHEMY_CONN"] = f"sqlite:///{AIRFLOW_HOME}/airflow.db"
os.environ["CAPSTONE_PROJECT_DIR"] = os.path.abspath(".")

# Run the full DAG end to end for a single logical date - this actually executes every task in order
run = subprocess.run(
    ["airflow", "dags", "test", "capstone_support_pipeline", "2025-01-01"],
    capture_output=True, text=True, env=os.environ,
)
print(run.stdout[-3000:])
print("STDERR tail:\\n", run.stderr[-2000:])
print("\\nReturn code:", run.returncode)


## Step 8 — Summary and README

A summary of every deliverable with its actual results, plus a `README.md` ready to go with the code.

In [ ]:

import subprocess

bronze_count = spark.read.format("delta").load("lakehouse/bronze/support_tickets").count()
silver_count = spark.read.format("delta").load("lakehouse/silver/support_tickets").count()
gold_count = spark.read.format("delta").load("lakehouse/gold/category_metrics").count()

summary = f'''
=========================================
 PROJECT SUMMARY
 Real-Time Customer Support Intelligence Platform
=========================================

Deliverable 1 - Ingestion (Kafka):
  Topic: support-tickets | Produced and consumed successfully with schema validation.

Deliverable 2 - Delta Lakehouse:
  Bronze rows: {bronze_count}
  Silver rows: {silver_count}
  Gold rows  : {gold_count}
  MERGE demonstrated on Silver and Gold. Schema enforcement verified.

Deliverable 3 - RAG:
  {len(chunks)} chunks indexed from {len(kb_articles)} knowledge-base articles.
  Hybrid search (dense + BM25) and cross-encoder reranking working.

Deliverable 4 - Orchestration:
  Airflow DAG 'capstone_support_pipeline' executed via `airflow dags test`.

Deliverable 5 - Quality gate:
  Great Expectations checkpoint success: {ge_result["success"]}
  OpenLineage events written to lineage_events/*.log
=========================================
'''
print(summary)


In [ ]:
%%writefile README.md
# Real-Time Customer Support Intelligence Platform

An end-to-end pipeline built in a single Colab notebook, using synthetic support-ticket data.

## Architecture

```
Synthetic Data --> Kafka (KRaft) --> Pydantic Contract --> Delta Lakehouse
                                                              |
                                          Bronze -> Silver -> Gold (MERGE)
                                                              |
                                                     Great Expectations + OpenLineage
                                                              |
                              KB Articles --> Chunking --> Embeddings --> Qdrant + BM25
                                                              |
                                                  Hybrid Search + Cross-Encoder Rerank
                                                              |
                                                     Airflow DAG (orchestrates all of it)
```

## Modules (`pipeline/`)

| Module | Responsibility |
|---|---|
| `data_gen.py` | Synthetic KB articles + support tickets, with injected quality issues |
| `contract.py` | Pydantic data contract (schema validation boundary) |
| `ingest.py` | Kafka producer/consumer |
| `lakehouse.py` | Delta Lake bronze/silver/gold with MERGE + schema enforcement |
| `quality.py` | Great Expectations checkpoint + OpenLineage event emitter |
| `rag.py` | Chunking, embeddings, Qdrant index, hybrid search, cross-encoder reranking |

## Orchestration

`airflow_home/dags/capstone_support_dag.py` defines the DAG
`generate_data -> kafka_ingest -> lakehouse_load -> quality_gate -> build_rag_index`,
run end-to-end via `airflow dags test capstone_support_pipeline <date>`.

## How to run

Open the notebook in Google Colab and use "Run All". Total runtime: roughly 10-15 minutes
(includes downloading Kafka, Spark, and the embedding models).


In [ ]:

print("README.md written. All 5 deliverables implemented and executed.")
